# 3D Brain Network Visualization
## MDD UKF Coupled Oscillator Study — Interactive Plotly Networks

This notebook reads the network data exported by `main_analysis.ipynb` (Section 7.6)  
and produces **interactive 3D brain connectivity graphs** for all four coupling metrics.

### Prerequisites
Run Section 7 of `main_analysis.ipynb` first to generate:
- `results/network_edges.csv`
- `results/atlas_coords.csv`

### Four coupling metrics visualized

| Variable | Formula | What it captures |
|----------|---------|-----------------|
| `w_k` | $k$ | Raw spring coupling constant |
| `w_norm` | $k/\sqrt{ab}$ | Normalized: coupling relative to intrinsic oscillator dynamics |
| `w_sync` | $k/|a-b|$ | Synchronizability: proximity to Arnold-tongue locking boundary |
| `w_frac` | $2k/(a+b+2k)$ | Coupling fraction: share of total stiffness from coupling; $\in(0,1)$ |

### Output files written
All HTML files are **self-contained** (Plotly JS from CDN) and open in any browser:
- `results/figures/network_3d_w_k.html`
- `results/figures/network_3d_w_norm.html`
- `results/figures/network_3d_w_sync.html`
- `results/figures/network_3d_w_frac.html`
- `results/figures/network_3d_all.html` — **4-panel combined interactive figure**

## 1. Setup

In [ ]:
import os, sys, warnings
import numpy  as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

warnings.filterwarnings("ignore")

# ── Paths (relative to project root; adjust if running from a different CWD) ──
DATA_DIR = "results"
OUT_DIR  = "results/figures"
TOP_N    = 200          # how many strongest edges to draw per network

os.makedirs(OUT_DIR, exist_ok=True)
print(f"Data dir : {os.path.abspath(DATA_DIR)}")
print(f"Output   : {os.path.abspath(OUT_DIR)}")
print(f"Top edges: {TOP_N}")

## 2. Load Network Data

Two CSV files are read:
- **`network_edges.csv`**: one row per region pair, containing MNI endpoints
  (`xA,yA,zA,xB,yB,zB`), raw coupling weights (`w_k`, `w_norm`, `w_sync`, `w_frac`),
  and their log1p transforms.
- **`atlas_coords.csv`**: one row per HOA region — name, MNI (x,y,z), lobe label.

In [ ]:
edges_path = os.path.join(DATA_DIR, "network_edges.csv")
nodes_path = os.path.join(DATA_DIR, "atlas_coords.csv")

for p in [edges_path, nodes_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"{p} not found.\n"
            "Run Section 7 of main_analysis.ipynb first to export the network data."
        )

edges = pd.read_csv(edges_path)
nodes = pd.read_csv(nodes_path)

print(f"Edges : {len(edges):,} rows, {len(edges.columns)} columns")
print(f"Nodes : {len(nodes)} rows")
print(f"\nEdge columns:\n  {list(edges.columns)}")
print(f"\nNode columns:\n  {list(nodes.columns)}")
print(f"\nLobes present: {sorted(nodes.lobe.unique())}")

## 3. Colour Palette and Metric Configuration

Node colours match the lobe palette defined in `R/atlas.R` so the 2D and 3D plots
use a consistent visual language.

Each metric is described by:
- `col` — the raw weight column name
- `log_col` — log1p-transformed column used for visual normalization  
  (critical for `w_norm` and `w_sync` which have extreme outliers)
- `title` / `subtitle` — figure labels
- `colour` — edge colour

In [ ]:
# Lobe colour palette — mirrors LOBE_COLOURS in R/atlas.R
LOBE_COL = {
    "Frontal"  : "#E74C3C",
    "Temporal" : "#3498DB",
    "Parietal" : "#2ECC71",
    "Occipital": "#F39C12",
    "Cingulate": "#9B59B6",
    "Insula"   : "#1ABC9C",
    "SCGM"     : "#95A5A6",
}

# Metric descriptors
METRICS = [
    dict(col="w_k",    log_col="w_k_log",
         title="Raw k",
         subtitle="Absolute spring coupling constant",
         colour="#2C3E50"),
    dict(col="w_norm", log_col="w_norm_log",
         title="k / \u221a(ab)  — Normalized",
         subtitle="Coupling relative to intrinsic oscillator dynamics",
         colour="#2980B9"),
    dict(col="w_sync", log_col="w_sync_log",
         title="k / |a−b|  — Synchronizability",
         subtitle="Proximity to synchronization (Arnold tongue measure)",
         colour="#27AE60"),
    dict(col="w_frac", log_col="w_frac_log",
         title="2k/(a+b+2k)  — Coupling Fraction",
         subtitle="Fraction of total system stiffness from coupling; bounded (0,1)",
         colour="#8E44AD"),
]

def hex_rgba(h, alpha):
    r, g, b = int(h[1:3],16), int(h[3:5],16), int(h[5:7],16)
    return f"rgba({r},{g},{b},{alpha:.3f})"

print("Palette and metric config ready.")

## 4. Core Trace Builder

### Edge binning strategy

Plotly `Scatter3d` does not support per-segment colour/opacity within a single trace.
The workaround is to **bin edges into 12 opacity groups** by their log1p-normalized
weight, creating one trace per bin. This produces a smooth visual gradient from
transparent (weak) to opaque (strong) edges.

### Why log1p normalization?

`w_norm` and `w_sync` have values up to ~3×10⁹ for pairs where $\hat{a}$ or $\hat{b}$
hits the positivity floor (1e-8). Without log scaling, the single maximum edge gets
opacity=1 and every other edge gets opacity≈0 (the single-visible-edge problem seen
in the original 2D plots). `log1p` compresses this range while preserving rank order.

### Node traces

Nodes are grouped by anatomical lobe; each lobe forms one trace so that the legend
allows toggling individual lobes on/off.

In [ ]:
def build_3d_traces(edf, ndf, metric, top_n=TOP_N):
    """
    Returns a list of Plotly Scatter3d traces for edges (binned by opacity)
    and nodes (grouped by lobe).

    Parameters
    ----------
    edf    : DataFrame — edge list with weight columns and MNI endpoints
    ndf    : DataFrame — node list with MNI coordinates and lobe labels
    metric : dict      — {col, log_col, colour, title, subtitle}
    top_n  : int       — how many top edges (by raw weight) to draw
    """
    col, lcol, colour = metric["col"], metric["log_col"], metric["colour"]

    # Select top edges by RAW weight
    df = edf.nlargest(top_n, col).copy().reset_index(drop=True)

    # Normalize LOG weight to [0,1]
    lw      = df[lcol].values.astype(float)
    lw_norm = (lw - lw.min()) / (lw.max() - lw.min() + 1e-12)

    traces  = []
    N_BINS  = 12
    bins    = np.floor(lw_norm * N_BINS).astype(int).clip(0, N_BINS - 1)

    # ── Edge traces (one per opacity bin) ────────────────────────────────────
    for b in range(N_BINS):
        mask = bins == b
        if not mask.any():
            continue
        frac  = b / (N_BINS - 1)
        alpha = 0.03 + 0.88 * frac    # [0.03, 0.91]
        width = 0.5  + 4.0  * frac    # [0.5,  4.5 ]
        sub   = df[mask]
        xs, ys, zs = [], [], []
        for _, row in sub.iterrows():
            xs += [row.xA, row.xB, None]
            ys += [row.yA, row.yB, None]
            zs += [row.zA, row.zB, None]
        traces.append(go.Scatter3d(
            x=xs, y=ys, z=zs,
            mode="lines",
            line=dict(color=hex_rgba(colour, alpha), width=width),
            hoverinfo="skip",
            showlegend=False
        ))

    # ── Node traces (one per lobe) ────────────────────────────────────────────
    active    = pd.concat([df["ROI_A"], df["ROI_B"]]).unique()
    sub_nodes = ndf[ndf["name"].isin(active)].copy()

    for lobe, grp in sub_nodes.groupby("lobe"):
        c = LOBE_COL.get(lobe, "#aaaaaa")
        traces.append(go.Scatter3d(
            x=grp["x"].tolist(),
            y=grp["y"].tolist(),
            z=grp["z"].tolist(),
            mode="markers+text",
            text=grp["name"].tolist(),
            textposition="top center",
            textfont=dict(size=7, color="rgba(30,30,30,0.72)"),
            marker=dict(
                size=5,
                color="white",
                line=dict(color=c, width=2.5)
            ),
            name=lobe,
            legendgroup=lobe,
            hovertemplate=(
                "<b>%{text}</b><br>"
                "x = %{x:.1f} mm<br>"
                "y = %{y:.1f} mm<br>"
                "z = %{z:.1f} mm<extra></extra>"
            )
        ))

    return traces


# Standard scene layout used for all individual plots
SCENE_LAYOUT = dict(
    xaxis=dict(title="x MNI  (L ← → R)",
               showgrid=False, zeroline=True,
               showbackground=True, backgroundcolor="rgba(238,242,250,0.88)"),
    yaxis=dict(title="y MNI  (P ← → A)",
               showgrid=False, zeroline=True,
               showbackground=True, backgroundcolor="rgba(238,242,250,0.88)"),
    zaxis=dict(title="z MNI  (I ← → S)",
               showgrid=False, zeroline=True,
               showbackground=True, backgroundcolor="rgba(238,242,250,0.88)"),
    camera=dict(eye=dict(x=1.45, y=1.45, z=0.75)),
    aspectmode="data"
)

print("build_3d_traces() and SCENE_LAYOUT defined.")

## 5. Individual 3D Networks

One interactive figure per coupling metric. Each figure saves to an HTML file and
is also displayed inline in this notebook.

**Interactive controls:**
- **Drag** to rotate
- **Scroll** / pinch to zoom
- **Double-click** to reset camera
- **Hover** over a node to see its name and MNI coordinates
- **Click** legend entries to toggle anatomical lobes on/off

### 5.1 — Raw Coupling Constant k

In [ ]:
m = next(d for d in METRICS if d["col"] == "w_k")
traces = build_3d_traces(edges, nodes, m, top_n=TOP_N)

fig = go.Figure(data=traces)
fig.update_layout(
    title=dict(
        text=(
            f"<b>Brain Network — {m['title']}</b><br>"
            f"<span style='font-size:11px'>{m['subtitle']} | "
            f"Top {TOP_N} edges | log1p opacity | HOA 110 ROIs</span>"
        ),
        x=0.5, xanchor="center", font=dict(size=14)
    ),
    scene=SCENE_LAYOUT,
    legend=dict(
        title="Lobe", x=1.0, y=0.93,
        bgcolor="rgba(255,255,255,0.88)",
        bordercolor="#cccccc", borderwidth=1
    ),
    margin=dict(l=0, r=0, t=90, b=0),
    paper_bgcolor="white",
    width=950, height=720
)

out_path = os.path.join(OUT_DIR, f"network_3d_w_k.html")
fig.write_html(out_path, include_plotlyjs="cdn")
print(f"Saved: {out_path}")

# Display inline
display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                          config={"displayModeBar": True,
                                  "scrollZoom": True})))

### 5.2 — Normalized Coupling k/√(ab)

In [ ]:
m = next(d for d in METRICS if d["col"] == "w_norm")
traces = build_3d_traces(edges, nodes, m, top_n=TOP_N)

fig = go.Figure(data=traces)
fig.update_layout(
    title=dict(
        text=(
            f"<b>Brain Network — {m['title']}</b><br>"
            f"<span style='font-size:11px'>{m['subtitle']} | "
            f"Top {TOP_N} edges | log1p opacity | HOA 110 ROIs</span>"
        ),
        x=0.5, xanchor="center", font=dict(size=14)
    ),
    scene=SCENE_LAYOUT,
    legend=dict(
        title="Lobe", x=1.0, y=0.93,
        bgcolor="rgba(255,255,255,0.88)",
        bordercolor="#cccccc", borderwidth=1
    ),
    margin=dict(l=0, r=0, t=90, b=0),
    paper_bgcolor="white",
    width=950, height=720
)

out_path = os.path.join(OUT_DIR, f"network_3d_w_norm.html")
fig.write_html(out_path, include_plotlyjs="cdn")
print(f"Saved: {out_path}")

# Display inline
display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                          config={"displayModeBar": True,
                                  "scrollZoom": True})))

### 5.3 — Synchronizability k/|a−b|

In [ ]:
m = next(d for d in METRICS if d["col"] == "w_sync")
traces = build_3d_traces(edges, nodes, m, top_n=TOP_N)

fig = go.Figure(data=traces)
fig.update_layout(
    title=dict(
        text=(
            f"<b>Brain Network — {m['title']}</b><br>"
            f"<span style='font-size:11px'>{m['subtitle']} | "
            f"Top {TOP_N} edges | log1p opacity | HOA 110 ROIs</span>"
        ),
        x=0.5, xanchor="center", font=dict(size=14)
    ),
    scene=SCENE_LAYOUT,
    legend=dict(
        title="Lobe", x=1.0, y=0.93,
        bgcolor="rgba(255,255,255,0.88)",
        bordercolor="#cccccc", borderwidth=1
    ),
    margin=dict(l=0, r=0, t=90, b=0),
    paper_bgcolor="white",
    width=950, height=720
)

out_path = os.path.join(OUT_DIR, f"network_3d_w_sync.html")
fig.write_html(out_path, include_plotlyjs="cdn")
print(f"Saved: {out_path}")

# Display inline
display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                          config={"displayModeBar": True,
                                  "scrollZoom": True})))

### 5.4 — Coupling Fraction 2k/(a+b+2k)

In [ ]:
m = next(d for d in METRICS if d["col"] == "w_frac")
traces = build_3d_traces(edges, nodes, m, top_n=TOP_N)

fig = go.Figure(data=traces)
fig.update_layout(
    title=dict(
        text=(
            f"<b>Brain Network — {m['title']}</b><br>"
            f"<span style='font-size:11px'>{m['subtitle']} | "
            f"Top {TOP_N} edges | log1p opacity | HOA 110 ROIs</span>"
        ),
        x=0.5, xanchor="center", font=dict(size=14)
    ),
    scene=SCENE_LAYOUT,
    legend=dict(
        title="Lobe", x=1.0, y=0.93,
        bgcolor="rgba(255,255,255,0.88)",
        bordercolor="#cccccc", borderwidth=1
    ),
    margin=dict(l=0, r=0, t=90, b=0),
    paper_bgcolor="white",
    width=950, height=720
)

out_path = os.path.join(OUT_DIR, f"network_3d_w_frac.html")
fig.write_html(out_path, include_plotlyjs="cdn")
print(f"Saved: {out_path}")

# Display inline
display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                          config={"displayModeBar": True,
                                  "scrollZoom": True})))

## 6. Four-Panel Combined Figure

All four metrics side-by-side in a 2×2 grid at the same `TOP_N` edge threshold.
Each panel has an **independent camera** — rotate each panel separately to compare
the spatial distribution of coupling under different weighting schemes.

**The SCGM cluster** (thalamus, amygdala, hippocampus; grey nodes at $z \approx -8$
to $+20$ mm) is of primary clinical interest for the MDD rtfMRI-nf study.
Long-range oblique edges connecting deep SCGM to lateral prefrontal or posterior
cortex represent the inter-network coupling most relevant to the treatment hypothesis.

In [ ]:
print(f"Building 4-panel figure (top {TOP_N} edges per panel)...")

fig4 = make_subplots(
    rows=2, cols=2,
    specs=[[{"type":"scene"},{"type":"scene"}],
           [{"type":"scene"},{"type":"scene"}]],
    subplot_titles=[m["title"] for m in METRICS],
    horizontal_spacing=0.01,
    vertical_spacing=0.06
)

scene_keys = ["scene", "scene2", "scene3", "scene4"]
seen_lobes = set()

for idx, m in enumerate(METRICS):
    row = idx // 2 + 1
    col = idx  % 2 + 1
    for tr in build_3d_traces(edges, nodes, m, top_n=TOP_N):
        # Only show lobe legend once (first panel)
        if hasattr(tr, "name") and tr.name in LOBE_COL:
            tr.showlegend  = tr.name not in seen_lobes
            tr.legendgroup = tr.name
            if tr.showlegend:
                seen_lobes.add(tr.name)
        else:
            tr.showlegend = False
        fig4.add_trace(tr, row=row, col=col)

    # Per-panel scene: no axis labels (too cluttered in small panels)
    panel_scene = dict(
        xaxis=dict(title="", showgrid=False, zeroline=False,
                   showticklabels=False, showbackground=True,
                   backgroundcolor="rgba(238,242,250,0.88)"),
        yaxis=dict(title="", showgrid=False, zeroline=False,
                   showticklabels=False, showbackground=True,
                   backgroundcolor="rgba(238,242,250,0.88)"),
        zaxis=dict(title="", showgrid=False, zeroline=False,
                   showticklabels=False, showbackground=True,
                   backgroundcolor="rgba(238,242,250,0.88)"),
        camera=dict(eye=dict(x=1.5, y=1.5, z=0.7)),
        aspectmode="data"
    )
    fig4.update_layout(**{scene_keys[idx]: panel_scene})

fig4.update_layout(
    title=dict(
        text=(
            "<b>3D Brain Coupling Networks — All Four Metrics</b><br>"
            f"<span style='font-size:11px'>Top {TOP_N} edges per metric | "
            "Edge opacity \u221d log1p(weight) | HOA 110 ROIs | MNI coordinates</span>"
        ),
        x=0.5, xanchor="center", font=dict(size=15)
    ),
    legend=dict(
        title="Lobe", x=1.0, y=0.95,
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="#cccccc", borderwidth=1
    ),
    margin=dict(l=0, r=140, t=95, b=0),
    paper_bgcolor="white",
    width=1300, height=980
)

out4 = os.path.join(OUT_DIR, "network_3d_all.html")
fig4.write_html(out4, include_plotlyjs="cdn")
print(f"Saved: {out4}")

display(HTML(fig4.to_html(include_plotlyjs="cdn", full_html=False,
                           config={"displayModeBar": True, "scrollZoom": True})))

## 7. Output Summary

After running all cells, the following HTML files are available:

| File | Content |
|------|---------|
| `results/figures/network_3d_w_k.html` | Raw $k$ — 3D interactive |
| `results/figures/network_3d_w_norm.html` | Normalized $k/\sqrt{ab}$ — 3D interactive |
| `results/figures/network_3d_w_sync.html` | Synchronizability $k/|a-b|$ — 3D interactive |
| `results/figures/network_3d_w_frac.html` | Coupling fraction $2k/(a+b+2k)$ — 3D interactive |
| `results/figures/network_3d_all.html` | **4-panel combined** — all metrics side-by-side |

The HTML files are **self-contained** — they load the Plotly JS library from CDN and
require no local installation to view. Share them by email or upload to a web server.

### Relationship to cross-subject analysis

The 3D visualization is built from a single subject (`TP_E3834`, pre-treatment session).
Once the batch pipeline in `batch_analysis.R` runs on all 23 subjects, the same
`plot_3d_networks.py` script can be called with group-averaged or $\Delta k$ edge weights
to visualize treatment effects across the whole brain in full 3D.

In [ ]:
# List all output files
import glob
files = sorted(glob.glob(os.path.join(OUT_DIR, "*.html")))
print("=== Generated interactive HTML files ===")
for f in files:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {f:<55}  {size_kb:7.1f} KB")